# 🔐 Nemo Safe Synthesizer Tutorial: The Basics

#### What you'll learn

In this notebook, you’ll explore the fundamentals of the NeMo Safe Synthesizer by optionally enabling PII replacements, training on a sample dataset, generating synthetic data, and assessing its quality and privacy.

The model can generate realistic synthetic data that mirrors the structure of the original dataset, including numeric, categorical, and text features.

The total job run is approximately 10 and 20 minutes without and with PII replacement respectively.

### ⚡ Colab Setup

Run the cell below to install Nemo Safe Synthesizer with the engine and CUDA 12.8 extras. Use the NVIDIA Artifactory index for package access.
Installing the kagglehub to load the sample dataset in the following cells.

In [ ]:
%%capture
!uv pip install nemo-safe-synthesizer[engine,cu128] --extra-index-url "https://urm.nvidia.com/artifactory/api/pypi/nv-shared-pypi-local/simple"
!uv pip install kagglehub


### 🔑 Set the NIM API key

Run this only if you will enable PII replacement later. The cell sets `NIM_ENDPOINT_URL`, opens a page in your browser, then prompts you to paste the `NIM_API_KEY` you copied from that page. If PII replacement is disabled (default in this tutorial), you can skip it.

In [ ]:
import os
import webbrowser
import getpass



# Step 1: Set the NIM endpoint URL
os.environ["NIM_ENDPOINT_URL"] = "https://integrate.api.nvidia.com/v1"

# Step 2: Open the NVIDIA page where you can get your API key (sign in and create/copy a key)
NIM_API_KEYS_URL = "https://build.nvidia.com/settings/api-keys"
webbrowser.open(NIM_API_KEYS_URL)

# Step 3: Prompt for the key you copied from the page and set it
if "NIM_API_KEY" not in os.environ:
    os.environ["NIM_API_KEY"] = getpass.getpass("Paste your NIM API key from the page that opened: ")
print("NIM_API_KEY and NIM_ENDPOINT_URL are set.")

### 📥 Preview input data

Load a tabular dataset (here, women's e-commerce clothing reviews from Kaggle) and inspect the first rows. Safe Synthesizer uses this as the training and reference data.

In [ ]:
import pandas as pd
import kagglehub 

# Download the dataset
path = kagglehub.dataset_download("nicapotato/womens-ecommerce-clothing-reviews")
df = pd.read_csv(f"{path}/Womens Clothing E-Commerce Reviews.csv", index_col=0)

print(f" size of the dataframe is: {df.shape}")
df.head()





###  ⚙️  Initialize the Nemo Safe Synthesizer interface:

Create the Safe Synthesizer builder and attach your DataFrame. You can always enable PII replacement by adding `with_replace_pii()` to the builder, which requires `NIM_API_KEY` to be set as an environment variable. Model training and generation is enabled by adding `synthesize().resolve()` to the builder. The `with_` options control the configuration—they don’t directly run the model.  You can use the [reference docs](https://aire.gitlab-master-pages.nvidia.com/microservices/nmp/latest/nemo-microservices/latest/safe-synthesizer/about/reference.html) for full config options.

Run the pipeline with `run()`, which executes process_data, optional PII replacement if set, training, generation, and evaluation in a single call. Results are available on `builder.results`.



In [ ]:
from nemo_safe_synthesizer.sdk.library_builder import SafeSynthesizer

builder = SafeSynthesizer().with_data_source(df).synthesize().resolve().run()
results = builder.results

### 📤 Preview output data

Inspect the generated synthetic data including row count and preview of the first rows.

In [ ]:
synth = results.synthetic_data
print(f"Number of synthetic rows: {len(synth)}")
synth.head()

### 🛡️ Evaluate quality & privacy results

The pipeline computes quality and privacy metrics. The summary holds timing and scores; the evaluation report is rendered as HTML.

In [ ]:
print("Summary (timing and scores):")
print(results.summary.model_dump())

In [ ]:
# Download the evaluation report as an HTML file
if results.evaluation_report_html:
    report_path = "evaluation_report.html"
    with open(report_path, "w") as f:
        f.write(results.evaluation_report_html)